
# RepairForge AI
## Open-Weight Model Evaluation for Autonomous Code Repair

**Goal:** Evaluate an open-weight code model on software bug investigation and repair.

### Experiment workflow

1. Load Qwen2.5-Coder.
2. Provide a controlled buggy Python function.
3. Identify the root cause.
4. Generate a minimal patch.
5. Validate generated code.
6. Run regression tests.
7. Record verified repair metrics.

### Core principle

A generated patch is not considered a successful repair until the relevant tests pass.


In [10]:

import torch
import platform
import json
import time
import os
import re
import ast
import math
import textwrap

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("WARNING: GPU unavailable. CPU inference may be slow.")


Python: 3.13.15
PyTorch: 2.11.0+cu128
CUDA Available: True
GPU: Tesla T4
GPU Memory (GB): 15.64


In [11]:

%pip install -q transformers accelerate


Note: you may need to restart the kernel to use updated packages.


In [12]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

dtype = (
    torch.float16
    if torch.cuda.is_available()
    else torch.float32
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()

print("Model loaded:", MODEL_NAME)
print("Device:", model.device)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-Coder-1.5B-Instruct
Device: cuda:0


In [13]:

def generate_response(prompt, max_new_tokens=400):

    messages = [
        {
            "role": "system",
            "content": (
                "You are RepairForge AI, a precise software "
                "debugging and code repair agent. "
                "Analyze evidence and propose minimal repairs. "
                "Never claim tests passed without execution."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    start = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    duration = time.perf_counter() - start

    generated_tokens = outputs[
        0, inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response, duration


In [14]:

BUGGY_CODE = """
def calculate_average(numbers):
    return sum(numbers) / len(numbers)
"""

BUG_REPORT = """
The calculate_average function crashes when an empty list
is passed.

Expected behavior:
- A list with numbers should return its average.
- A single-element list should return that element.
- An empty list should raise ValueError with a clear message.

Actual behavior:
- Empty input causes ZeroDivisionError.

Task:
Identify the root cause and generate a minimal repair.
"""

print("Original buggy code:")
print(BUGGY_CODE)


Original buggy code:

def calculate_average(numbers):
    return sum(numbers) / len(numbers)



In [15]:

TEST_CASES = [
    {
        "name": "normal_input",
        "input": [2, 4, 6],
        "expected": 4.0,
        "exception": None
    },
    {
        "name": "single_input",
        "input": [10],
        "expected": 10.0,
        "exception": None
    },
    {
        "name": "empty_input",
        "input": [],
        "expected": None,
        "exception": ValueError
    }
]


def evaluate_function(function):

    results = []

    for case in TEST_CASES:
        try:
            actual = function(case["input"])

            if case["exception"] is not None:
                passed = False
                observed = f"Expected {case['exception'].__name__}"

            else:
                passed = math.isclose(
                    float(actual),
                    case["expected"],
                    rel_tol=1e-9
                )
                observed = repr(actual)

        except Exception as exc:
            passed = (
                case["exception"] is not None
                and isinstance(exc, case["exception"])
            )
            observed = f"{type(exc).__name__}: {exc}"

        results.append({
            "test": case["name"],
            "passed": passed,
            "observed": observed
        })

    return results


def print_results(results):

    for result in results:
        status = "PASS" if result["passed"] else "FAIL"

        print(
            f"{status} | {result['test']} | "
            f"{result['observed']}"
        )

    print(
        f"\nPassed: {sum(x['passed'] for x in results)}"
        f"/{len(results)}"
    )


In [16]:

baseline_namespace = {}

exec(
    compile(BUGGY_CODE, "<original_fixture>", "exec"),
    baseline_namespace
)

original_function = baseline_namespace["calculate_average"]

baseline_results = evaluate_function(original_function)

print("ORIGINAL IMPLEMENTATION RESULTS")
print("=" * 45)

print_results(baseline_results)


ORIGINAL IMPLEMENTATION RESULTS
PASS | normal_input | 4.0
PASS | single_input | 10.0
FAIL | empty_input | ZeroDivisionError: division by zero

Passed: 2/3


In [17]:

repair_prompt = f"""
You are debugging a real Python defect.

BUG REPORT:
{BUG_REPORT}

ORIGINAL SOURCE:
{BUGGY_CODE}

Return exactly this structure:

ROOT_CAUSE:
One concise explanation.

PATCH_CODE:
```python
def calculate_average(numbers):
    # Complete corrected implementation
```

Requirements:
1. Keep the public function name unchanged.
2. Preserve correct average calculation.
3. Empty input must raise ValueError.
4. Include a meaningful error message.
5. Do not change unrelated behavior.
6. Return the complete replacement function.
7. Do not claim that tests have passed.
"""

response, inference_time = generate_response(repair_prompt)

print("MODEL:", MODEL_NAME)
print("Inference time:", round(inference_time, 2), "seconds")
print("\nMODEL RESPONSE\n")
print(response)


MODEL: Qwen/Qwen2.5-Coder-1.5B-Instruct
Inference time: 7.28 seconds

MODEL RESPONSE

### ROOT_CAUSE:
The `calculate_average` function attempts to divide by zero when given an empty list. This is because the length of an empty list is zero, leading to a division by zero error.

### PATCH_CODE:
```python
def calculate_average(numbers):
    if not numbers:  # Check if the list is empty
        raise ValueError("Cannot calculate average of an empty list.")
    
    return sum(numbers) / len(numbers)
```

This patch checks if the input list `numbers` is empty before attempting to calculate the average. If it is empty, it raises a `ValueError` with a clear message indicating that the list cannot be averaged.


In [18]:

def extract_patch(response):

    blocks = re.findall(
        r"```(?:python)?\s*(.*?)```",
        response,
        flags=re.DOTALL
    )

    for block in blocks:
        if "def calculate_average" in block:
            return textwrap.dedent(block).strip()

    raise ValueError(
        "No valid calculate_average code block found."
    )


def validate_patch(patch):

    tree = ast.parse(patch)

    if len(tree.body) != 1:
        raise ValueError("Unexpected top-level statements.")

    fn = tree.body[0]

    if not isinstance(fn, ast.FunctionDef):
        raise ValueError("Expected a function definition.")

    if fn.name != "calculate_average":
        raise ValueError("Unexpected function name.")

    if len(fn.args.args) != 1:
        raise ValueError("Unexpected function signature.")

    if fn.args.args[0].arg != "numbers":
        raise ValueError("Unexpected function argument.")

    allowed_calls = {"sum", "len", "ValueError"}

    for node in ast.walk(tree):

        if isinstance(node, (ast.Import, ast.ImportFrom)):
            raise ValueError("Imports are not allowed.")

        if isinstance(node, ast.Call):
            if not isinstance(node.func, ast.Name):
                raise ValueError("Attribute calls are not allowed.")

            if node.func.id not in allowed_calls:
                raise ValueError(
                    f"Disallowed call: {node.func.id}"
                )

    return True


patch_code = extract_patch(response)

validate_patch(patch_code)

print("PATCH VALIDATED")
print("=" * 45)
print(patch_code)


PATCH VALIDATED
def calculate_average(numbers):
    if not numbers:  # Check if the list is empty
        raise ValueError("Cannot calculate average of an empty list.")

    return sum(numbers) / len(numbers)


In [19]:

safe_builtins = {
    "sum": sum,
    "len": len,
    "ValueError": ValueError
}

patch_namespace = {
    "__builtins__": safe_builtins
}

exec(
    compile(patch_code, "<generated_patch>", "exec"),
    patch_namespace
)

repaired_function = patch_namespace["calculate_average"]

print("Generated patch loaded successfully.")


Generated patch loaded successfully.


In [20]:

patched_results = evaluate_function(repaired_function)

print("REPAIRED IMPLEMENTATION RESULTS")
print("=" * 45)

print_results(patched_results)

baseline_empty_failed = not next(
    result["passed"]
    for result in baseline_results
    if result["test"] == "empty_input"
)

all_patched_tests_passed = all(
    result["passed"] for result in patched_results
)

repair_verified = (
    baseline_empty_failed and all_patched_tests_passed
)

status = (
    "VERIFIED"
    if repair_verified
    else "VALIDATION_FAILED"
)

print("\nFINAL REPAIR STATUS:", status)


REPAIRED IMPLEMENTATION RESULTS
PASS | normal_input | 4.0
PASS | single_input | 10.0
PASS | empty_input | ValueError: Cannot calculate average of an empty list.

Passed: 3/3

FINAL REPAIR STATUS: VERIFIED


In [21]:

evaluation_report = {
    "project": "RepairForge AI",
    "experiment": "Single bug repair",
    "model": MODEL_NAME,
    "inference_time_seconds": round(inference_time, 3),
    "bugs_attempted": 1,
    "bugs_reproduced": int(baseline_empty_failed),
    "patches_generated": 1,
    "patches_verified": int(repair_verified),
    "baseline_tests_passed": sum(
        item["passed"] for item in baseline_results
    ),
    "patched_tests_passed": sum(
        item["passed"] for item in patched_results
    ),
    "total_tests": len(TEST_CASES),
    "repair_status": status,
    "baseline_results": baseline_results,
    "patched_results": patched_results,
    "patch": patch_code
}

with open("repairforge_evaluation.json", "w") as file:
    json.dump(evaluation_report, file, indent=4)

print(json.dumps(evaluation_report, indent=2))
print("\nSaved: repairforge_evaluation.json")


{
  "project": "RepairForge AI",
  "experiment": "Single bug repair",
  "model": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
  "inference_time_seconds": 7.281,
  "bugs_attempted": 1,
  "bugs_reproduced": 1,
  "patches_generated": 1,
  "patches_verified": 1,
  "baseline_tests_passed": 2,
  "patched_tests_passed": 3,
  "total_tests": 3,
  "repair_status": "VERIFIED",
  "baseline_results": [
    {
      "test": "normal_input",
      "passed": true,
      "observed": "4.0"
    },
    {
      "test": "single_input",
      "passed": true,
      "observed": "10.0"
    },
    {
      "test": "empty_input",
      "passed": false,
      "observed": "ZeroDivisionError: division by zero"
    }
  ],
  "patched_results": [
    {
      "test": "normal_input",
      "passed": true,
      "observed": "4.0"
    },
    {
      "test": "single_input",
      "passed": true,
      "observed": "10.0"
    },
    {
      "test": "empty_input",
      "passed": true,
      "observed": "ValueError: Cannot calculate avera

In [22]:

from IPython.display import FileLink, display

display(FileLink("repairforge_evaluation.json"))


/kaggle/working/repairforge_evaluation.json